In [1]:
from pyspark.sql import SparkSession, DataFrame
import logging
from typing import List, Optional
from delta.tables import DeltaTable
from delta import configure_spark_with_delta_pip

logger = logging.getLogger(__name__)

builder = SparkSession.builder.appName("delta_app") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.2.0, io.delta:delta-storage_2.12:2.2.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")


spark = configure_spark_with_delta_pip(builder) \
    .getOrCreate()




In [2]:


def write_delta_upsert(df: DataFrame, file_path: str, key_columns: List[str] = None) -> None:
    """
    Loading data to delta table through upsert

    Args:
        df (DataFrame): Dataframe with data to upsert
        flie_path (str): Path to the delta table
        key_columns (List[str]): Primary key column for merging
    """
    try:
        # full_path = f"s3a://{bucket_name}/{file_path}"
        full_path = file_path
        logger.info(f"Writing dataframe to delta location: {full_path}")

        if key_columns and len(key_columns) > 0:
            key_conditions = " AND ".join(f"target.{col} = source.{col}" for col in key_columns)
            non_key_columns = [col for col in df.columns if col not in key_columns]
            update_dict = {f"target.{col}": "source.{col}" for col in non_key_columns}
            insert_dict = {f"target.{col}": "source.{col}" for col in df.columns}

        delta_table = DeltaTable.forPath(spark, full_path)

        delta_table.alias("target") \
            .merge(df.alias("source"), key_conditions) \
            .whenMatchedUpdate(set = update_dict) \
            .whenNotMatchedInsert(values = insert_dict) \
            .execute()
        
        logger.info(f"Successfully loaded data to delta table {full_path}")    
    except Exception as e:
        logger.error(f"Error upserting to delta table path {full_path}: {str(e)}")
        raise